In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config


In [0]:
v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")
print(v_data_source)
print(v_file_date)
print(raw_folder_path)

In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

races_schema = StructType(fields=[StructField("raceId", StringType(), False),
                                  StructField("year", StringType(), True),
                                  StructField("round", StringType(), True),
                                  StructField("circuitId", StringType(), True),
                                  StructField("name", StringType(), True),
                                  StructField("date", StringType(), True),
                                  StructField("time", StringType(), True),
                                  StructField("url", StringType(), True) 
])

races_df = (spark.read 
.option("header", True) 
.schema(races_schema) 
.option("nullValue", "\\N") 
.csv(f"{raw_folder_path}/{v_file_date}/races.csv"))

In [0]:
from pyspark.sql.functions import to_timestamp, concat, col, lit

races_with_timestamp_df = races_df.withColumn("race_timestamp", to_timestamp(concat(col('date'), lit(' '), col('time')), 'yyyy-MM-dd HH:mm:ss')) \
.withColumn("data_source", lit(v_data_source)) \
.withColumn("file_date", lit(v_file_date))

races_with_ingestion_date_df = add_ingestion_date(races_with_timestamp_df)

races_selected_df = races_with_ingestion_date_df.select(col('raceId').alias('race_id'), col('year').alias('race_year'), 
                                                   col('round'), col('circuitId').alias('circuit_id'),col('name'), col('ingestion_date'), col('race_timestamp'), col('data_source'), col('file_date'))

print(races_selected_df.schema)
races_selected_df.write.mode("overwrite").option("mergeSchema", "true").partitionBy('race_year').format("delta").saveAsTable("f1.bronze.races")

dbutils.notebook.exit("Success")